# Exploration - EDF Tempo

Analyse des données réelles (RTE éCO2mix, Tempo, Open-Meteo) stockées par le pipeline.
Pré-requis : `python -m src.pipeline run`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from src.processing.features import Inputs, build_training_frame

inputs = Inputs.from_db()
df = build_training_frame(1, inputs)
COLORS = {0: '#2563eb', 1: '#94a3b8', 2: '#dc2626'}
NAMES = {0: 'Bleu', 1: 'Blanc', 2: 'Rouge'}
df.head()

## 1. Répartition des couleurs par saison

In [ ]:
pd.crosstab(df['season'], df['target'].map(NAMES))

## 2. Température du jour vs couleur
Les jours rouges sont les jours ouvrés les plus froids, tant qu'il reste du stock.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for k, name in NAMES.items():
    d = df[df['target'] == k]
    ax.scatter(d['day_in_season'], d['temp_mean'], s=12, c=COLORS[k], label=name, alpha=0.7)
ax.set_xlabel('Jour dans la saison (0 = 1er septembre)')
ax.set_ylabel('Température France pondérée (°C)')
ax.legend(); plt.show()

## 3. Thermosensibilité de la consommation

In [ ]:
c = inputs.consumption.merge(inputs.weather, on='date')
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(c['temp_mean'], c['conso_mean'] / 1000, s=6, alpha=0.4)
ax.set_xlabel('Température (°C)'); ax.set_ylabel('Consommation moyenne (GW)')
plt.show()

## 4. Couleurs par mois et jour de la semaine (règles Tempo)

In [ ]:
print(pd.crosstab(df['month'], df['target'].map(NAMES)))
print(pd.crosstab(df['dow'], df['target'].map(NAMES)))